# Industrial boundary continuation: one checkpoint before session end
Resume the verified baseline 200 / boundary 6 recovery, retaining the complete original 200-epoch schedule.
No per-epoch last/best saves. Save full recovery once at the session boundary (target 11h45 from first cell, including setup; leave 15 minutes for output persistence), or at epoch 200 if reached sooner. Kaggle may interrupt earlier. Resume from the newest saved output in the next authorized session. The completed baseline is verified unchanged. Distribution defaults to probe only.


In [ ]:
from time import monotonic
SESSION_STARTED_AT = monotonic()
from pathlib import Path
import os, sys, json, shutil, subprocess, zipfile, hashlib
PROJECT = Path('/kaggle/working/boundary-aware-finet-industrial')
roots = [p.parent for p in Path('/kaggle/input').rglob('train_industrial.py')]
dedicated_roots = [p for p in roots if 'finet-industrial-source-november-13' in p.parts]
roots = dedicated_roots or roots
if len(roots) == 1:
    shutil.copytree(roots[0], PROJECT, dirs_exist_ok=True)
else:
    archives = list(Path('/kaggle/input').rglob('finet_industrial_source.zip'))
    assert len(archives) == 1, 'Attach the FINet industrial source input.'
    PROJECT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as z:
        assert z.testzip() is None, 'Source ZIP failed CRC.'
        for m in z.infolist():
            assert (PROJECT/m.filename).resolve().is_relative_to(PROJECT.resolve()), 'Unsafe source path'
        z.extractall(PROJECT)
expected_runner_sha256='74f95c6c7ab67fdee4a557a8327629bcbd3c02bfea160fe3b40e04062766ec08'
assert hashlib.sha256((PROJECT/'scripts/run_industrial_kaggle.py').read_bytes()).hexdigest()==expected_runner_sha256, 'Update the dedicated source input; an older notebook output must not replace the verified runner.'
# Kaggle remote-input compatibility: concurrent checks retain every original hash/path check.
import base64, zlib
io_updates=json.loads(zlib.decompress(base64.b64decode('eNrNXH1z27jR/yq4dFqSjkzZTny9UU6dyTm5m0zvkjyJezMdWcOhKEjiiSJZvthWVH33Z3fxQoCkZKXpH70XWwKBxWKx2P3tYunds7KIhnE6r8uqiMMkmIdV6OfbZyP27I7+fV3P44rPWQQ9sg0r65wX93EJLb/9fssj9voNK/MkrgYMqPCcw4+0YtmCVSvObj68YfwRRsQbaPUFxbv0Ll0UQCvKkoRHVZylJYs3eVZU7Car04oXAzbni7BOqnkcVbp3GtVFgXQWdVUXXA+6XRU8nH/MsuTtI4/qKivuUvloFZarJJ7p73+UWaq/ZKUknYcV9lL0PsJX3akI03m20V/LVV3FiRxXwbLUIFjkIogE+3fNHGm9ybcsLFmay0Ef3/2qxrzbhEuuaGVFtALBgjTU4wrmLhdZsSm7ffzmoaaGU+dZEqJEf8vmXAsa9hj7x2mcLmmH1ZCf373n1a188gYelBw2chEnPEDJDdgmTOMFLyv6ivRuXt++/eXDp3dvP7Mxc51ZVlUJdwbMicKZ+pCXtfpY5LzCT8sinuPvVfiFJ2lNbQkHOfPCGdylzPrH2fAKVFF2y+Mkwd9lVPAH/ADyJ+pVllWrWVGXK/qG8ohBRwv89pBlNN+XOIeNcby79M2H316/ew9MOz+HEfTasv+rQ9DbLcitzIUaOnfpx08fbj/cfPgVOwqVDxqVDzb3FY+C+0sHRXEHqr5gIZ6PICviZZyGiVsAU95IrAg/Ax1UJ9Hug9JmyT13PdmBR1kxL6HPZCpaYENZFFZ8iQzGKWvEPWqkFC9YCpSJJhs2/YckhTh18NMSJeD5cRnM48L1RraQoVvJ2e9hUvO3RZEV7sL5LS5L0AKmliLPt6I+Yjv1ce94DbWa5BKWa1wGqI9rPMPVKHIBWQlckyu5xE0D1XLavMV4KogaqCifuz3rtIl6/jLJZq5zNjzz83TpeF6LoJCXoDtqa9tpwmgWP9zZs+8lYd9pTYurp0cB2hdc+UEOQJFAA2HJTX8/D8nWpeGGdwfAklqCHY/17oPFUhS/G0tN6Jm0d+nOP1I02BGafEEjvueCLVwBGvVssYgj8BUg1GITJudZmmyZMi/CGXRkgf+gkghRjFmP6i4LMJ7zoCrqaoUaLFcwZAtnZ4ilrPhmTwpHe92d5iGGGciy+hm4I7cZ66EdLrO6iPgBcYiHfpKFc9c7ILHlDPhPcz8sw6IIt64cA+7pnheV63z65aeOBmr68RdOik1D8Fu3XxUWS16JOb7wIitdFztOLqcDGj+5mHrgH6ttzsfQpY7T6gevV0NO1AF5PPT2oM1AF9AxGieeGTkt0gOjocnunQNCaW+YHvH0fp24Z3KVhthRKPj7CN3Dq9yEVbSC80GaNUR22RwBDvrlEpbcaNzBNVsb7R7Rpl9Bl9jf2IUHPXDLXb3l7IxdXV8f3VIxgx+m26/dy9cpjA/RCIQzccylDQAcAPaF8U0OnvPkLW6chB/O580ONw6xZ2AVp1v7qAklQUgjvov1kVuFvXTdFz+8HDD4AedDdP3Ey3ADBild+u/fvv709vNt30TUhwfxHGZbOMLHN/Y+aNv7YCc0HD607VKPOZI+3g9zxMbuzgEgNGqmJJQkZoJm9RFaxRwB7jk8EN8Gx7WVsBM6fGW39cDGBgBleznQx244ZRKhpTDUUUGARZjEAk8N8RQccek9D6pMYqGwDPKsjB9d75QZNW3UHSBtqtBhyn1WkPEEtP19lvKvmlespVyFV9ffw/waI5v+5asX0kfPMH/fzH0eP/KkmURGQ75ocCHkI9fi+TyNIF4AeT1H/+ZX2WwL0AxOpb/ij/MYQEt12jYhueBhBXMlgMUFddAMgEJc+ndiqQzSUCoo2DJXHmRh58p6AxOfMlmXKpx9TRKth0Fwbxz88D6ME4xXNHDd2daIsJtCbe5TYKWDP/cWVm/PBnvZ2MOncfk/0jyMC7DEYs5zmpMJUsBdY6g06II5E/Cj0vJ4OOH1i+uXxlQ907xVqE9D3kWdJDTwXJh+ddQhQERG2M6cpJm94BCcp8ruNYGSEUYBtK1iivvVeIA2nM/Hl1cvXl5/r3yVSBf89WJ4eQ3/sdnWsI+aS4F+DciLFvMVCSuHJtRreFRDLxxc6hwETkCdOJp9I93gouZ6TTgmOERRS14NOUoCE/FkYp+36VQZffFY0oRThxH40TkF9zClnMC/x60qLS8OKgUqj7rrFpPGiQBKhK8tGy8aTZcyFTpe6ElsncWtFROQ+lyCvKltV/hwUN1+K9YhKcY+peLOm/YusQhgbwkxoFxVDB+BcihACerNK5aCvpSYXIIhoAAFv4/5gxV3CEFPIJ6p3LjihVqQp/dFBpggGnDK0xb3ntbnEnYJ5SziK3DaJRiZCcjUgW2J58SR2YpRrf6+b3aV5h8I6nR25fyCUR943KDNNc8pwOmxTEH5n+gXHNUdnpX9v3dEz8JcgrSPdE0sDGTA5teLBYB60cV4+Cf2Woiq3ggovuAPvIBIL8Rwr+DcSuwJ4AKwvAq3IP60AtME4d+KJ2A56srvaJGcj/3IXrQVgQQ7acQ19WGrcF8w5u05ByW10UO9STZJgM1g92sjqiLGxmAuH93LAdvEYLHQeLkma2fMv7wGN2U1nrNLjw2H7MprJTRktrOsZyUX+Qy3rQdqjycjnB7DNbdZpX6Iz0ZXZ7LLU/7ONbRPkxCjR1PP65ct8XpErHIRh8QqTfluzbcjpaxkicRJh+YBo+/COuKUSov3jeHPC465DCOP5mrzga4Vk70lbBvJb2Cc+XGT/er1D4epqKyb1cNIvw3EY2NEJzdX8kT4wzGrwC5xZSwa9rzG3WKwpQegMOk74B/Zhnk4IWu3WZN3zCE7N6ssg6YQaKUi2pKbJVPr4BC1D2x40gZQJ25lwsUUDuAXUoxAdfIxLe7o1VhjQXnAN9mupwzvSTAYcwfG5YEa6FoUrJ3xLANBhLTzAv7BCoPLENhRyQ4FKjuiEogu+OlJx/IWWceURMOjxAs1epd5vFhwTLIZAhwi4VcsktKHjVyBMZyDlaOkseVg5PEg3rTwjLVq0RE8wSjcfIguiVKzXT34GYIB4r2UC/kIyglnhzNCTmECFleeKvCa4Dte4YJQV/iDtdOYgRTZbQaLnnFEzMh2scWQWK0F6AVNKrqV0LaPUCdvXfhRlm9dw/Wb1EAeFjYhmyP3VWa8GyyIibGnMaKBkpTJcp13myVaReeXWyub7Npaj/09f7NGqYvsajm+LWo+YLRRQbamrydiPwGKOyeLWMGs5d2znYKFiC72GBoo4NnkrYCAJWEYqUa1wulpMxT2r6X7WekncbqW6aOBYM5QVf4Y8bxiHz6TQrUGizst2kjK+x0i8hDP8TSveLxcISBSjKqgb2qd7c4yKHKftuZuJ/+eFIagQqlBSj91/eYM+hXbVl6NEsb/QVYNw+3WHJp+k6AVIhkICR3Lzpo58JbigO6eojbtHJjgBqJcsEJG7sC0UkjNr3MwFFxkK8b008qcWK7QSM0Qx+P+bMuhMYegjJlCGbcSKHKm7tNmSS3PRoi8jFZ8EwLQBnDnAKqn+AFhGCBlvAsZMXHvh/eIRVZlUZZAm7rkw1tF9Cgjsiw9fJueadRyS3QlmVF2zgGbhw7Z6K3gb+PFxmPsQukbZ55FNd6Ig/kWXt3pm1xdtAQznkYr2O91wBM4BnjROmI/h0AKmdhkax7gJYxu7KElzrS4oMa17JrM3cjKFuzNRF6VVcr+lEEoKgFkf8PMe73MKxrK+jdzU8Qrb/rt2LXXi/QmgySCyTPYcVy4o9MEaFGAgCI6VIyc04ihCILPRZagmx9QEfcrFaf/+AJ5aQKc4zullCwA4ECa8UFdluk8OgYCZiiLQRRADFlfoe7RfHa7Ap/PNsBfgoEwDmBREsab5gIO4AlQ3J5r/ZAgvPTZB4UNwyJa4RVewfFAYLHApq4wGyXuJ43QzrzOa4o2yv4Vg7h4WtISb27YT/88f39z/vk1e+lfvGKrqsrL0XD48PDgUzYdDO9miBAGmTmveAgspcvhXBQclEPqdB7Oe2eSWgAzaRxwdtagg71tFSbOAkjzIi/A8oJCje0SBo1MpTGRgRs34Uari42GHwo4A0GFOQUEzf683uTNEFEEk1bjK0ylOnd3eMtNyVVgauzU1eL8h1aeTA1t4qWjPI06KF0kZLR1o5yLrmKALbb79dhIGiHM5NGI5D0oYIhHJiHc2S0IMlC2ntTpQMZmp9TOStwQ0+NdJxsju+2t5GYsE5um6TrK/RudXBIpDPbujWYOrD8F1gPxCRWF63KGgVnVIMNeo4xBBuTdFEAT8ZuMIQa1pADUpp3SDtFL5txwemrw9IrFVwqM9EP2F70OMwSVD38co3yfjpgw0hw09nDA6nSdZg8USmQAnhJQoL6IygqMtABxM+2caGdbZYxAW07RgWLZTkhKkn8xd+jkxKKA6pgcQjuX8HBdQjBUPQCpzkLsOg4lUQWfBGut54Ib1UV8a3IDelO+a29Bl+HPFJzOM1gpmXuUOOPwY8uk9xVrIR6VvTgYhYvg26xIMrATWS8cWrqUCyGI5WMhnbBuHavlHbA8pr0lBT1gcI8u/F07QDeoshiTpXS+9N6cYLefrL4iWwtebrGVSEPl6Ud2wk+fdxGHwcmmGKSdcpOo3rUDFzzenZlbp10VXXRuMvHQNUhY3Al+p+OuhYOJub0+WqdVNhmSpuEsWoUg6jlAQkn37lk8v3s21YnlP7G/h8tlgqF2XlfEUAnnETDFCqIeLHw8xzYGzIPSbH32d85zqbXI8xC5h2l4tB4oigAIwAaDMUZsRY9KIz0BfhG2f0BpEzit99JmI2b49P4XMElVVmMFhq9uVMJCZOrMIkz3dM9DQWi3ihQ06jF4yIo1L8rx5RXFnDk8b+kHzTdgASoJTwHXF2gGsKO/CXPXUrEB05iZXXYK31r3dn+DiOHi4oLE4NIs7M/sGhoglLhgamb8Zg7rq/Sgc2QClp0D+wMHduQYhgP68CXgmy3C2CXoLLkxSTegdcSI/8WCn76gZVijiQGEhnAmnxR8gU3GdnvfzuWG7qHPIqnLlZmW6YVNAEnKkjWqLUtZ3b76ViUfPP4BrBmiiADDtIUwlYMGZhIiCOslwuAxRVZWEhSBj+v5msbTo81LF53Sq3PgmYcb9umXnyQGj7/IXPg/b2Y3BXtzc0s6sCj4v2o8YqjxFaY2o9I3aTYCIBQ/AnLi0gkOBtbBwvcYY8aoyHJcbaWzhIAc4ecfcQUbYJF8L4IW/ijvfTYwAV3+yBIg5IviLRFXyQ5o1WZ0jZfj/Wr4EG5902kmC7/4/eckzinfSCJqPV5Wga5wxhS8rnb2b8B0ZCV3J7YCGj0+dcuBYrM2etyplNbVQQOTzm12CzEO2AFvarhbZYqCB0r6lG34KcJc4FlFuD3Xs1aRjnkVq2gpO/n2XzWGjHiK4BBns3AWU80ybqauR7irLy7mf5W6M9SBprzUGwjzVvA8CSMu6vAldQgxAT3AyQHYA0JHR1un0h+8oluGElOsAjAX9BhxPnL7mnb+J986lJNL/wILJUgCk6fX3UWDYJCfDdgzhM7lUABordPWawmftynwByeAqciHAmuAvqD4xDs4vkX8KF4UgICFpeiRRDwNzimGMC3iJ7yR0PciQr5tnvIcv7deMOgv/1dvCmBtTWUV57dewFDUVe6qyVe1lW/QnzIf9AGk/vl04a5+i0A9usnSRYxeeAm+YIm0NhweRBQm0ZkV4Y5sbU6IFLsy+/rq6Olyd/OFAlHv3wZjssiD+qqCNuiJFZkdRKYSHI+kYKjV7tVFn3tEZajwWGH9n1n51xT6IZnTy/sUwYPVfZ0aPFEKf0rVU29h4NN1f19fLWYs4kDRmpbb0UK1/ROFQXTMDTUWmT2wV0U2ryPMdgYKLgZgdAKyawRFejSro3tfe8uk7sRK8M6VNe7QQEoLF2hd8Gpbw7CDhL476Z5r/LIv3/2fpLhPS6CdEFHJpUwQxTW9JmvpwNZfmRWZokgn198DEr7C/w2bgOEyEGsm6SuAkuzsDmSQ+/MLQHmP8+46lmbf0kidA1HFhTIzTDqIOQS8zCj4H5TeP10TmyInYOLsTD6dXGDhENkOR/dw9hZNZWP082/UbSqMWoutG9y35C3LOJqLPGSxEeM93WecSsFYkhj8v6bZsyJb85Tu5IQ/b3fQl7Xmku0F9yzUPiSCUvN6x1FKazITNKvF40S0Tf2Cb7J77hoz2v3EPLrUrduPUKHAIj7lCUq3SRTgbRy41LFDyTLLs/aJU8zptU7QLExCAFlgsQWADDCxhTcjAaAwjImpiDXIigBB6ClHiD3vtk1GlxfSdAicbFdWLpIstMo5AQ+LC2yQ+Zc4N6zuIWBvrp6mOAnaPh/LiaxDFyaJKKHC8yOlD1tUZI/upTCjuqCKptKGry1cCr+kBNEibUKIthoWFCrTUhXhmnlp/QKs7osr87aaZAVNFsd9WM/lIk9Mv9Qd+izLEs+bOCB27pBxv2ypfLe72WHycvTDgMEPvLXBiP9kNuR7IObUF08PVm/H6AoAtXyDYJzV1lKK7IHw687EdYYraaEwUV1ydnYC8/sWQjtxBnkLfHyKnm3ayyNT1puNqGbowHwXF9vSBdF74rS1TV/yHtwBPZKHaQA+SlW1mJvmX7eU3IhTwmAGMBAjEtD3Amvp6SUsrd+HX+RuJ4MDerstCIy0rHzZspxcSrGEdoaccjtDR85vvi8/CyFyjlMuSujMBLoCgLOTSWFhKsjoMCkpy5mfZzmGSqJ/kISb2Tx0yCX7l6LY7EgXtTGiS1ZXeV3hS8KO90qRNhtt84UEZqKpHSm6Z2ffLDLPV+7F/So31Vx/Gra6w6C8cNa39OPm4l5epTelFo7NSpP9uedzI1Mq3ZH13vfxd+4Pv11/En7EiWQN24mFaoQwQP7WJcbROwzaOoiEWxcCwQ7J7P1ZnDrdcfJqXLw55Jo1TM9xWPOSkdcuVRYXJdAHDT+R6r70bPR67jS3ieP2/ciJyLIHWPbhSoUNusEzGNyjSPvbyhLMfThSq3tCRYJnB9/iSrBbcyCtrdJqmesvA7rFEfYSQUYuk9ZloMFSkS7dapPLSt9Nlq75NsdDaZ2Gw3/qQ94q2n9sQ4+r6FLGSE/9mkVrawj9TQyrpS+hhtWC2bxOeCO4gaklh461WtnQQTLKGOrC67Gk6mstlDlggHmAKBNglrJdA+LbvHgE0xCtwX3hQOtqVL1togtwqYdZhAmqgYRbR5emVGgfx9Ah6lYriwmkdja75YNdDKuqcMWCBo5ekTOQvCrd3FYrOEkFXtCR8APrLRXAMng7QUX29BhbsEsgmwUVSozKkQC9msF2N+l2pJSP33QbfzWB/Fyj5QYllZQncVG3phBtbWQPeu4KzSSHbUVtZjuCkFBfSk1cmZBkOGb33V4pDbRoFfGoqknjnpaW6ArxOsXoStD4HafGolisHA3E7LLz5XTQ9LycypJ18exq1KIEDQdNh8iIAEYDtFYUdV4FeC8s/qCAOk0dR3kg/31ov/8rh9g19GXYs92T88vpxJHlxpZzmznyjuS0UFpClHdmdVS5ZpqG+U7fyQr+pPyxc8DLKMw5WvE0QCYFakazTV7nv7sf3wJ7vtUg11UZzzE/0zyXbQZaUS32XqqxEhy2X6BpaYRdR+34/tCcZ3CgxFn28bxvRgUnuvuv1UhZIfWNKvls//8oV1FN')))
io_prior={'src/industrial_data.py': '694d5d866488fe51899efb2af5712c98591abf49de2dfd31ce69ea01f13e1cf5', 'tests/test_industrial.py': '9c279ae7f51e9e9cb4dacdd44b46cbad0947af6746e21f2c94420182c2b40361'}
io_expected={'src/industrial_data.py': '62db09839658547583e3e8c5dd42f82318e6bed0f9938117d71f4bb332421fa1', 'tests/test_industrial.py': '5656ab3459240d40d32d86377a9ae3afa24dcb2651384b317aa62565b3803235'}
for relative,content in io_updates.items():
    target=PROJECT/relative
    assert hashlib.sha256(target.read_bytes()).hexdigest() in (io_prior[relative],io_expected[relative]), relative
    assert hashlib.sha256(content.encode()).hexdigest()==io_expected[relative], relative
    target.write_text(content,encoding='utf-8')
print('INDUSTRIAL_INTEGRITY_IO_UPDATE',json.dumps(io_expected),flush=True)
# Exact tested checkpoint-policy/source update; original model/loss/settings retained.
import base64, zlib
session_updates=json.loads(zlib.decompress(base64.b64decode('eNrNPYuS20Zyv4I4lQCQuViS+5C0Eq+ik+2LKj5JJduXSlEsFAgMubgFARwAarXe7L+nu+eNF7mSnMR3tSKBmZ6Znn53z/D+u7qKT9M82ddNlUZZ2FRRmqf5NijvvrtyvvtI//uFlVEVNczRDR32uWRVumN54+xr6ODs8/g6yrcscX5685Y1TpQnTlGl2zSH1tBpHzf7ijlZUdcBB/sx/5hvqmLnJFETxVlU16x20l1ZVNC7TtK4mehXH3Px5u91kasvu6i5FkBK+JilawngvX7TwDTl412RF02Rp/HEgflvwrjY5w2rPmqQ+X5X3sH4Tl6K/u/f/Cy7v9lFW6aaNkUVq0Hwc7Bv0qwOcNKyxw/w+eciSlg1cf6TpdvrhiUfADfF7pdoV2Z8aIIAOxGwT1G2j5q0yMMdA0THCiFrmGgSVXfhZma036Q5a8IoiUpYhGxas4zFTZiwT2nMJoB72LoQ0RnW19H84tLob+y8Nel3f3315u3Eef/h3a/vXr/7eeK8UQ1xQTWDrclgVaEBYBfl6YbV8KbGlQFNhLe04NoYsCzqpqyKmNVENGK8CroVFQvhzTpap1na3CGUT/iE4czTwsSSJFLZ3ftVPHhd5Jt0O3GismR5EmYFfm6KXRqHtD8hwpw411GV3EYwXsWw/+Rj7gz+t4tuWLgrEpaFRQmUlP6OOyknHF+z+KYs0hxWXe3zsE63eYR0LqZvvB8bpGYsCdknVt0117AM6HwH3FQBnf4OkGi5Yd2wcuLcVilsJvKA34MQaxM3acbC66hWFIqNFI1IhgyRIZEGP+b/ZjAb/WPsOsdsC9H+FV8U9guromiuEK6zcFx8cvrXv/3K4lc/hPUeWO1TWsMaP81c3qUGAmkUyRzR79TuESAKBKxi35R7oPe0UnD4o9qQbKdroFogShYismfzs/OLSz0dIJgsjTnjAWvCIiUozgp6nUD6SKZNEReZbCPZRCyN87WaC/vHHrgDgLNtAfybF9UOvkdr8QFoMItihpLUmM9+iw9oQgrQNYjT3wt4moWbLC3DIs8AHIuQGAEx9Y2BXhIBZuddFFcFyIQNPkfKlPMB9mJh1ITT4MJFKqCFso0DkiiFBTMPYG3kThNs3BbPD1QDX79LNw61D7oIdf5JIhPUgkON2vjEJjYuzf9kL4FfbPxFuO2DaeKbAD8C171zlPgnYEfh3kQx/gecVjPnb6AP2I9VVVSe+1t+kxe3uamFJeZeOEnhgGZzamD6vMnuHK6MnQhE1TYF2QFy1NDZgetznsedJhFnCPKM1FXtxUKaasG+ZTkDQ6Co5GS5TFl01YPoHCjRoMEs3YrFRZXU7sp8yKVcyh+L/Vj8Wu2ZIC9BbojTbzKehscHFcMABQMGPZqN77x0BNx11KD6AHlsbFN3i/S8HKWkSHA5ae3UQJJIuA1sDfxhDgF1ECjthyE8YIm9xoLXVq18ooFYoT8xJo9aSpE/YdLYwIXeSj5wxUAb5I6n7RUOZuLopS86yJjICS/Ev2CxVUUZgvYQQw6qPbCzwtuiugFKk3CNR71TJZutRmoGfpZ9f4qyemSYMkVTagfcJkfhdlEAZlHV1Ldpc+258T6JXN9vQTFQoYnFwsdscnAZ9fV+s8kYn6ZvcF3NtMiRxp5nGUAwwS0aWs01SJ3rIksWwQVQMItq1NvSIOQ8IqhSm0tAQQYw508LDUawLkGHZnkZRHVUVdGdJ0dMmruSLdZFkWmuMKAFYEeWDEUb7yC+g/BDEQTw0hoMUzBUzPX4AVC/5x/LPnolp2KmO3gFptsnhlZ/jHaSw0dxEpRpNTStFR+laNXXTKICvnqegZx/FTP3g3q/83zRqSlA3ovWujFv4nyvFmv02Of98P+7F77gsXsX5b575cydJ/ZET8UUAN38AwOicWZBizDdtNhD91ZXPhfoyj8MdN1FOPIGZHzj4c6vLaIDSsCtxzfU5PLcd07UWoD0cq/DJq7hmgBo45uBEU3Ncx/n2KZiPt23Rd5mZVfAACOwTD9DK77wzv6AuHP5EHY7a9MehKHLnbU034BazGNu36MZY9o9hkb0yP7nHg+JOOFZcStzQbMGovwcciFY0xNJ6SRnQPkuLIdTGk0Emdw++aQqbusJucngDS+cJSit5Yq/2gCHwazYZyGE4IvDQNqgdGQen53JYIhnPStUQcifODnyzgkUSgajUcsIWVcsujE4drvGhdY34EOB+sFp3uIc0WtlQVN4HDM+zZQe4hRprisNRRjn0JFW7xFUBc+wJjkSAu7NebiFttvi0RB8Rq0xxRh+APJBMyBnQlSVMDjHlzRCpQpdElaM2aKG4H5/UMA8vFa3dLeVfXyMGtTFvopZC40pdoch+csA9AT4eo3nfvjLn13/6KG2zeGR+qR6e9SfQdU5f3KmuqepKRZ93rgn8LmcAjnSagJUgCZaiRLu3TQBxuO4XOIXaO9Ke9d4ox7h+7TWxrHZ23y86qr4J0+O16Edrcn18YO1AEVqlUWGwEZ8+QMc9NI5a+1CK3ThEcYmjjUzDFAJtPogvS2UjcUKHEByua/KokZcmSrTsJCTNNrmRd1gnAtnfM2y5AQGcxrYW9zgfdbU7gRQCEt90Eoe2yIijtXSxpCAG7YrQYVIipYBrIUTbbfghaBMlXuEY0iLV4c6pPhoaUub6U3tRQqJSwlSAKAMizzBJrasBR0mxDAuWUziQdtiAxMUWMijHYlij+vtCde/E65LJ7b+87UDjZOrvW1V7EvL6hHLumF37ZUsqyU8XZEkQ0nvUGfcF/HCoL+VKdMMSo3yO6/bugXS1+qW3kBzfEcLFSwhGDSlld8/aAUkORc71AXi1LsHbJkszUcDgQBNEJEP5vr5mkBpVHpS2Igv04SyWKjBDHksnfsWCANVLcFh9B3vScQ/0lujZKnmRTJP6E4gO/S+OIYP8DCZNmAQhyQasKtJLsd01uGEFgiJILKzFLaG7Kte2CJ8wgGHqJZCoKsQUMZwgjnKeFTHgK2OebZC1aKxK+by6FEFMkV3QX8qsFODsOTmEWn8ZRcbqyXn1pVtF+g9DOhZ7RGaBqGs2rLI2mqSEiABQNqEhpLTg3Ssb9ihEFsTfLVlHE7HUh+JGnWkYAs1vrX7El+DRjZJbh7XIjMhR5GO6uUHGOv0TfHb6Z9/mqFtDwYKtUXXF2NiDooQ/mQaOH999aOzR7u1im4NbYfYdrt2fZGAWA8p0pxuRLAQx3yV351oUnNIitdOVDFDrdVcr0XKXE8chakTARqexCm6hYFryHoj0IVq2sO80cQRFr4R4q/ja5bsKaRhBCA6MbF+uo4LJBOY1WKKCRhW4r9rHjtkwmGgr7Qx4gEmavb1wlWRahlCUrHBVk7hD547n7met5yhUDxldIcGKjAhd6fwi5xRVIKu5ju6cOMStaUIWFEk1YzrCTDBvqRgcjdwvJDZKBkuRjW1cClL4Lz6wfFi2E9MhKh0AQ+5+W7/AttR54XOcmnGXehAoX44ZJkVGyBfJKg1eJPXu6i6CWEDt+laxn3kZsuosNh2TQH9cE1SJRu+Xtzj4q90bscjMzIM6UHogxdTFxlQtR+UwDDAN8vZCqxL7OUfFsHiP5SY2AMFpue2MsSIvqC8Q8NnOHfs+tKo7KTfPLHhwgJGgjHDYphFM9e9K26YikQjPdaLuWQHEW9rJyJA8FFDMMzno4bsqwakeAQGcXNbSA8BiAeHFBBQ6FTsH/sUXugYrZ2tE7NTlh+asOjrmVlYz4o9SgNZkJjwRftyqbKfnfySrKyj3b6ZCcOMFKJWNNLpMY0f/ixgn9O6QRWIzgxajuI5OM0VtAe/uYO/n4DQfqRuXW+AYw6WmGBUxYmyikXJncNHeeHE1wU4LSCzc3YLohw8HqAK7S+IsXc3OLKgXhG2JghhcWPKDSXYlAD6i3wC1A+owswQbpXCIXz2jbTFxE4pfEkWxIjh9MtfCbeVQFZALXrAYCaMnP4upyS28xQcDutVUDbXruxjazO74VfoBqWNbIiupiERR8d4IZrqPIhu0AvfFHyKUgkFLotuRCw+ROi17e/9s/PjZ5T9KTqoQE3CcP1eK3agAxHKIh3I7Q6K/tREwZRgEc3rgEPNiuKG/I37ijvXV45h+/fkiKT/cw20ipuwzNlnMLNQHJpysTdxRdkjGnCJ7VZtZ6JrBj8SONo8owOsrEz112TKOAKUkk6Rlmm2KBzMNFEiC0KMxMjc9+2ggocQQBctp6sABQ0XOfhGvpiJF+MZgg9IGXzBp4JAuNThVJCkHCjY65iYhMWglHGyaA2mL+gi5+zZuRI42ppEEaKKLLwuz+AcddgXJH+6uQu5wVIfSHPxMKqMEGTRbp1EKo9mP57odjesylnWaccfK375wAN1lEsE1oCFtkTImm3oPawj30qG4i5MYiQoUdUFZjya1JGuG/IMA68jYI4y9ZYul4AaJpIY0mVsJXm/WpSKsiS0Khb2VCU9NkprECHzXKkOvVPIvRVupxAFptM9QpVVDDEUvToc+ifCoDC+wViGWtFR1VHaJBBfTJx/CGUeRAwsS9r75BwRlr8Hl1IE8BY2vk4EIv1jorVlhZ4RlgcFyX5X1t69y9ChRc+ybVgSTnUYdAI+9b6+NglSO1uWPua99csRjWyWiH1jT41wRySp1TWfGYZ5XRVHJVMmUVVrSU8lG7D7lzOaXqIasl0eZ6HhsTMycTNIy10LBoyB4pZvtLmlIuA50UEhLvQMecfTRrWVreYyEBGLLg4Wq1YYvAAlKdNQclkWJArFwrsGHM0pVSPCv+2ZUAdKW0k02DBc+uaueqYlzQZjOdS4p+mxq2hvSf9caGHcChlYHoeuSdyCgyZjd6etJoMZUoln8yHlzvrzugYKsAIq5iKZ5x5JeqJRopOXQqQu530Pz8yHhtQB5bunyLKc3DeDjBucsQ2yDWpV3ODf09KTK5mIoR+x0Qa0gZ0DXoJue+mHGPpnnHW/Thf9cQrJqPXiuS9YVF+eX6/OLPiRyzEz/HNFi1ROS1kAksCoZkpEd0IRkrgoKU3H8m2aM8AH2ltksKJxhEFMjl2WnGApIyZg4yxKd93YlSurlgFcq4BZk5LL0QBNeAG9jE10oVnVyph0Nb93nF5KrlEMYgMrYBUpWQxfK8/BfN4Nk7W8SFGI7pqBrJa1NlF9hCJSXbhACHnpe1+bnsWC16CKQ/NEeEzkVl0JR2ei8g/ipfYOoI2RT+0AJwWMGKTCDVcrur5VGqqyCwnzQXwhDcvroqpDSZAhmfcZJhd4RMRVXKeF52Dr3oE2xd7oIfL7IRXU6lGQW3Ymj3OL6cqSEF34Br9ZaOzwYberDMPhQFFyR+OFOq/bSZiqnO/KTicuZ1erld+DZFUrT/tGyRhgovhapHRk7aS72Wc6pBkKCxQaUUBXBAl0Nb7Xtg05Z/Iy9YkQE3Zimz8bjHzKoXXw04BL4hd2xqg9uob9rBdPqUQQvubslq+LVyMdkLsbPKXD/TURsK7xhAbsk0HOCWwI5iWs8qaBGOwjwqCtUNI/9YSSunGAnws8/tFTbSuitTWfWAXE67z+7YdXL5zX73/DFAFirzZEcODagQqNCRWhmDovNYadlwvn2ejUfqMYJykijCZzPDrFBsMPO+AE55lDkMyRW6PaG2gm8kdH/gmhnBAUhQ+qmhRVQRgnQLIWm0bRbhrCWe8TrJIzZjSy/wPz6SJQFITiIS1dEjoC2Of188MDA+6nB8vkX0kQJwyzh9qBU2AUjWw0wlDu8jC4UxZ1SuFGdViMjpDpwpZvHrdXIfMFHW+jSEvNUzsm12N6B8ZrMFgAzYsENnjh7pvNyTPXiLd5ElwAe+pJu8QnzhKmCSLabmXbA9T4gElg74PYaxuoMlkmzv2Dz59xXkfjyPV7irA7UCXEpYa24pC25Z5P1Ig0Y/0nH4HShMqeHs8DORwtzl9QRpBoEHYdqpTmGkubRHhNCRzJQqr4qZMmEvwgtlwaKgY9AuxWHm+wrU/r1KjoN6dWB+USSgB9ULMVLcRwRaJihtdGnbWxboWiVt7miKQTD8yi2uKHQ79dDoq+tCYK1MuF35qJUcGszzFt5xTwB1U3WFnHJfX+F1NT3zQCytetnDqV199mxRqnoNP7PAutUv0wFPiE8H9uX+BftY0cqBkDJbpZ9EV9eNsvD3zZVZg0EOf9bpkAFwW8VuCgjvjANvzUclNIkoyjXbHfZFioK+LXWGTbY2IEppDqRyraPTjXpaveC11OYW/+ymjvGlVmA9shOvXVERgw6bVhZWOFogEbSxOIjoyYKF++S75ja65X5vKs+V5ZJNRnW7oY0laWtYKDQV43Y1FFVjXWrWMpmQr/oPjOgEmMulx9ltewsfkEybrO3IlY11gQWLToifB+21xwr0cguFz4ArZnPtav78hrhzd6e0pVKfsMBAxUJukVlm1VJ6jDALI2W8mEQtpCDZeQkAU1yM/bQ3tuPGLSEAVnIKG9Qd5Z10W2b0wjStuPeZztE0ZWJFsXxQ0M1+zLU6wYY9zWkl6E6LEYt0ctA5qK3NSodHJHWe9Pzi6nU+fEmc2nKlJNDq6wtBeaUoVlhs+l/+mYKnjUUV05T8xjhN87s2dTeRykQPGwS3Mp/PnQEz2ymLB2AHx5ksPwCWiVFgQdU+bbopJWVkFXUbZOiPRZ9aSL1YsTC5svbZw8mQVzXN98On505HAuzDjyYjxEadWVD3Z4VB2GIXS3hf+jU2hHhiAPRDIfnTEjy8HQId8vnJn91j4PQ0XXVBG/8jtLluk12qsF/UXRa8BfWEaAJZIXWdWC2CuEed5YymA7CyeIy1zPvzgXU4z6twnlEUk7FY4BSbEF1q1Hc3f6PC1SFdF1j6TQ6zH0pizzhccbrmzCe4HEq+lZ8mAc/dbHHEZOjvWFk8UY5mSNNAgQqvUGbfKFHGx5oDh4ZZtN1BnYu+Ww95tGf9OKkMxnjPRhLUWlKy/QN+YF0pYtlO5gW7gpqI0YOS659vTgT8bbPrOHGJ56bjRI2qrBftxEQpmqa1U1quya69WSzm2sSKiq3qbw4wrkQPcDcQ8ugkEu0ikGU0y3j4a10sWaHPEMNeWVv3neWGXDesSCWe6r8W3b4WpXZIRMr/3o1SHs/4era+v7oXNEPR4AWgd8SPOujN0OT7eSxd22hUdtaiNQPX7qqRcJHRM7A/wZAo5OHhCBT5x+n+LKOVSdbJy2srD2SNtdoGjMeJdNemR8j1ko3VEKZA9FWAetnpZB+NLY35eDjPx/xcSjLv0wkY8gDVH7+FoaAQ/3WsM86FKOse8oZwy6lqMQK7YTDAXcnGbKfTCyS/LRiWHw9poWKot8f8hrLrMoz+2XptU/nJ6SQEx4HWXyKLfcSE26dC9QH536ExNT/bM+Gd5Dd4i2cMSBV6PJNNGHI1yn0/BbO52Gzz7m302c7+q4SsumPuXMbHry+qwmygPrlrsPPA5kRHxk4xfcBY4aodbn4EOCji8wYm+ES2WqJwP+bow77uSVdtW2jKpaXx+XFq1L7YbvsZO3dt3VY3fa4WAf3r37VcZix4+FfMwBXFBScibH8hCPzipVHoLgRzJ67rkTqA1uou0WYO/SrQyB87ZxBNtrGOpRireg8HuaqLinBaeV9eRgFTBNKeDK3kxUqCH8PS31LWwZlpc0h6+1G8rW9Pds3zLXvgxtMhQmmgylcseusIujko44V7kEfMylc4+6VO5r75HTKklcK6cvkALLW11BgzSOXruk9+BVtd3jMf339MZLGN97LBgOw6SIgUDNrkGUJGEk+njuyQmFhShL51K1E1Uac1Q0IJLd8d4ykkWGmxbsAIouuSGragSP1ywrF+5bGajaADk3JzHLMiOwRaBf6LAWRbMoHgFSYTY/wdATr0aMMDUmJwyzJBOTz5v+wZnXXsu20aETfBtYj0M9CzzvOfq+4+JoJWflaKaYZzVDPiedqbyEZVEsbTTj9IsQiNRLJWPEmc1WdlWiCxURAG+lyHsCgXIq38upYFjv4snl1FqLkRpM6zD6FKVZtM5Yzz1EH9Aq2OmUIFhkexwRqwco62Xk+F5g3RYWFGRYhtBJDPByTiGE3VMu007xSihodUrXo2phohNQmSxNtOWahpDmoBVdUS4qqX7ERNM2b6gEpxyGIifyS6+Ng5qBDv9/uhIromCIukOuPr3/9HCqDAd+zpofGZQnWM3bEVwZ1E/SLaxKQtYZ0FJc4TJxiHdoeFnF+mCdFZV9zXOnx55EODgG1hYSV/Khlnoxq4n5VC5rZRW7tjSWF1GhKxoMlN5cygK91dKVCSHRwAIjcGQOTmesNsnzebI5v3z29HzKnj99ulnPLtnzdRxfbs7Pk028OZ+fr9mzZ/N5Mru42Jxdsug8jufT58l5sn5+Hrmjk137eBvQJU11fXCqGD7pS3Bho0uDvTpLUpjjS9ok84v504vpeRyt44tpwqbzy2k0Y/M4mc428ezsEl4/P3/Gnk3nzy/W63WczJ4+fTq/PLt4/vzi3F5StOwvG6Sx1kMv2yC6pY6y/8Abdz57fnY5i2B7Lmdnm7Pz9frscgM7wxg7nz9//izZsOnZ5vx5FJ2fXUYsiafx7PLZOtqsp8+fTTdyY4zSEqMQxJPs15sRMqtCfLv02tLXnkoc4YwHKjeHlymvBQGiqekqI7r6wxZxXEDBhNAN8dzjL3JtzZuPAsLjVdPAR2Fdi0uowVr5neFNDjkqD35VHFq1Ce8WyFth5d3T4nRNKUxfEgAlvwuC1gJ0bOC6tLDZg50BxPn64B1mkXqsYE/OCNYlEQaSpzSPUHOJjUBGE300yrEJwsEapXEo+NY3a/ysw4TiPuAnT+6fPDFExYRf4Uu1TcLHJDjHXP9hT8PoPUj4x0DVhSgCIsl8UwoJukDHV9QmXfEKxKupe8wI3WodMZLmWqyXOZUjnvYc5lXH5Q+VU3KVN1JNOVrH8k2LSzqIPKb6ZfB8UAfaVxeNGD6n1z3hM0Ex1z2U2NO3/xwM9TfLokdh9B9wIRi6fPsADMND9HTZAcGoeIUyHsuxqxrCrypm6JxOFPlsZeZSpcsfdTrxDzo0+yuqV9AyRuEbHgGjY+lk1xtHNU7pis0dS1LkZKN0D61WUaOw3m82xD5pEfz5Dlb15p2n7hNF6qeI8D2nNXV8wSYDTUjQYIDeumjSpGPV0Ji9HiZifr45Waw3u/GmNjO2LnLhLR93lYtJrRKqYDLFz9b0RvtpnKwmA0gZ7a+xs5oMoMe2OvB+WsMdIYH0aUXWkjReP620m0MNjvDC7JAjd8e25Z6uV2ShPiuE5axXY7Ws02GF56rTWDIsvu6vPbNj5/32O3jUwwOJOLOIxOtL9cmSDlVtKZ2Z6fgww2AVkrSjOgZRa4lBiBy9iBV+05bLWZyK/l84QElFm6FrRxSFur3+MBDvuEqSeyB5buKYN2hr53FCPqYpImi7+R1cZDkaARkK6fC1tPKrPO4t6a8oNmiVD5gc3TMph+n2lx9/+eXNu7fhj29/CF//+4+v/+P9uzdvfw3fv/v5zev/ckcQX2RpjGeP3LeFUSWGYrB+QfeF4RmEEz5VGViXsXNGvxrDs3U62h6MDCfjQRR151VeMPZsjifW8LqsHWAQ7wBP831DF7fNLsaAUSpIZIiovww9fVF66NCZHtq3Sb9dioD1sZyjDk8O5/fU2R111sISfXzCy57s0wq3Q74eTO7YHnRbiBoSwJKmxnN9v1nPhWJtu/ARKmkg0kEDyTPItLSuHDzMIYN5JY0+WTA9gNyhDaX4bC98Ff4kvFP5U/9Jup6sX9+BxC7Z9VJ0BBuCZT8LKwXDXRzDn5UxTmPmqjoU2rv+o5CqdYFarJhHZ46Urdo4IWnKMKSYTBhiYiIM5UkynqbgGUIMrNan+DfsuN90LMzMDL5WGDzhwo2u0a35mfP6Lm+uGV4iswe/DjD7Ga30Wp41wKuPcNL6ZEzrx6/G8n3Wb12VdzzLZSfldG5wIHsV1c4R+SfrrM8jfllIpoBEDMZrdmUobyXMb9hdyVWfRuyEX1tf7ctGHjW0zhWCOwe8fWOdMRQOMwoGAd64EytYp7lrtQt4pGSN+PLWGLHmt0r175TbDnKocH4n2iHOU55NnK7Pv0AZLh77k6Ov2tPhCQJgrg+2BLPxbTcomHWPod279rFsVy31BJd6QtET98G2Fcyx+IkeyzCARgKRFAuzmLXvqPtjD5QfEVkxTrzf98RmpMncXu1f3v/mPjwicnP4UPpRsz1wcjmYPfgqDKNYA1wxwGUjPGtK07rWkVlYJt94J7yyoz8SD6NQlXmJ2LL9CQtyHw6Pm+1QeFGDfxJetWMzBzFgBbGsmZJgOApK69CDBQU55rOgVFQhD8eB7AuRmQtFax3VVHhzix+vHM/42YP13xl39/Un+rEN9xfhQwAkfHjv8jJ3LNtTEw6m0+n8cqIcMBshD77n+49YQjccZG+Xt6Q7xvGnJyTYGNzjWhw0QsMVCInd6l+o4L9SoW8/p3jQE4kPjg4z6UrgZCl7q2gbFDkpgYGDxtQd3y+nK6ySP5tO7fJUqWAokYXHMGgwnonrrX+2c78tPaHAGdcfWndCi/rMWTB96P0BtVB4jaH8/ZDQ/gEREy0SrxIzokXPqIeuh8aLALr3Td+LH4ABaaQme4BgdHzP5TewHkdnPdXwriqEP5KDW3WVmka5ifGE8xg/FigQ5ulbZQN+LxI2Wl6u/Nav8HBIV1rdH+mrGT6ZYdUMXNB1jCPW4SltVZF9aoSNyIfmHyN4gTdU8JP7/XaXJCxYWR/jHrDZBvxZ73jPUdbT9vtXyI5n9k1R3P0DkeKZhbNnvu1e6iVAS31Alf9dtbBncm/Ii/UERnnRMpk4dZgX+hfVjkVleBwOTYvXwCk5D9ymD0gK1Z4phib89PViXBqZ4aDO1mh0tpECHyLYFe1gcbcmTGv81RLoDB3/CJLiP7pi0w9Oe8Q3HieNGXCPZ54l4A+MFvP2gzY1YdSXzOD2z2Xwh7lant9ComRuXUcsuJP/piMOGbaofhynXOfBEqfB6jhVbhSzSXEmFeNxAtY83jAM5Mvlh+HczcCACaR3R3+PFy79wmOuzp8cjLC04B0Zxxoguvlx8kg0uk4xKoxHQJZGmQG/7oTuduMnYh2FxlN0pU6tUxJWiQe/VAS71V5rJks6E8ij+9avsIhJENqWM1jCqlfJcJEopGSHrkHrIHGPE/EXkOw08L9CuCoCwxrA4FsKF5IUFpbY5xLL/zRqkoJ0RxPKKyXwl0NKKo0R+ahvjSxgo9m3wRcx5Nfy4LTDg4/mLXj+3cP/APOy49w=')))
for relative,content in session_updates.items():
    target=PROJECT/relative
    target.parent.mkdir(parents=True,exist_ok=True)
    target.write_text(content,encoding='utf-8')
print('SESSION_CHECKPOINT_SOURCE',json.dumps({p:hashlib.sha256((PROJECT/p).read_bytes()).hexdigest() for p in session_updates}),flush=True)

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
import torch, torchvision
assert torch.cuda.is_available(), 'Enable a permitted GPU; CUDA is not available.'
print(json.dumps({'torch':torch.__version__,'torchvision':torchvision.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0),'free_total_bytes':torch.cuda.mem_get_info(0)}))
# Retain the installed matching GPU torch/torchvision pair. Do not replace CUDA torch.
subprocess.run([sys.executable,'-m','pip','install','torch-dct==0.1.6','timm==1.0.31','pytest','scipy'],check=True)
subprocess.run([sys.executable,'-m','pytest','tests/test_industrial.py','tests/test_domains.py','tests/test_boundary_loss.py','tests/test_training.py','tests/test_industrial_pipeline.py','-q'],check=True)

subprocess.run([sys.executable,'-m','pytest','tests/test_industrial_final_only.py','-q'],cwd=PROJECT,check=True)


In [ ]:
subprocess.run([sys.executable,'-u','scripts/resume_industrial_boundary_final.py','--probe-only','--session-start-monotonic',str(SESSION_STARTED_AT)],cwd=PROJECT,check=True)


In [ ]:
RUN_BOUNDARY_TRAINING = False
if RUN_BOUNDARY_TRAINING:
    subprocess.run([sys.executable,'-u','scripts/resume_industrial_boundary_final.py','--session-start-monotonic',str(SESSION_STARTED_AT)],cwd=PROJECT,check=True)
else:
    print('Probe only; official training not started.')


In [ ]:
p=Path('/kaggle/working/FINetIndustrial/experiments/boundary/last.pth')
saved=torch.load(p,map_location='cpu',weights_only=True)
print('BOUNDARY_SAVED_EPOCH',saved['completed_epochs'],'of',saved['config']['epochs'],flush=True)
print('CHECKPOINT_SHA256',hashlib.sha256(p.read_bytes()).hexdigest(),flush=True)
print('RECOVERY_ZIP_EXISTS',Path('/kaggle/working/industrial_recovery.zip').is_file(),flush=True)
